# Master Training Notebook for 5 Image Restoration Models on Kaggle GPU

This notebook trains all **5 Image Restoration Architectures** on Kaggle GPUs (T4 / P100):
1. **Model 1: CNN Autoencoder** (Deep Residual Skip Connections)
2. **Model 2: Denoising VAE** (Latent Reparameterization & KL Divergence)
3. **Model 3: Perceptual GAN** (Generator + PatchGAN Discriminator + VGG Loss)
4. **Model 4: Restormer** (Transformer Architecture with MDTA & GDFN)
5. **Model 5: DDPM** (Denoising Diffusion Probabilistic Model with UNet)

Trained checkpoint weights are automatically saved to `/kaggle/working/weights/` as `.pth` files.

In [ ]:
import os
import math
import glob
import random
from pathlib import Path
from PIL import Image
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torchvision.models as torchvision_models

# Set seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {DEVICE}")

In [ ]:
# Directory Setup
IS_KAGGLE = os.path.exists('/kaggle/input')
DATA_DIR = '/kaggle/working/clean_images' if IS_KAGGLE else '../data/clean_images'
WEIGHTS_DIR = '/kaggle/working/weights' if IS_KAGGLE else './weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

print(f"Data Directory: {DATA_DIR}")
print(f"Weights Saving Directory: {WEIGHTS_DIR}")

In [ ]:
# On-the-Fly Degradation Transformations
def add_gaussian_noise(img_tensor, std=0.1):
    noise = torch.randn_like(img_tensor) * std
    return torch.clamp(img_tensor + noise, 0.0, 1.0)

def apply_gaussian_blur(img_tensor, kernel_size=5, sigma=1.5):
    return TF.gaussian_blur(img_tensor, [kernel_size, kernel_size], [sigma, sigma])

def apply_low_light(img_tensor, severity=0.5):
    scale = 0.6 - (severity * 0.5)
    gamma = 1.5 + (severity * 2.5)
    darkened = torch.clamp(img_tensor * scale, 0.0, 1.0)
    return torch.pow(torch.clamp(darkened, 1e-6, 1.0), gamma)

class KaggleRestorationDataset(Dataset):
    def __init__(self, root_dir, split='train', image_size=(128, 128)):
        self.split_dir = os.path.join(root_dir, split)
        self.image_paths = glob.glob(os.path.join(self.split_dir, '*.png'))
        if len(self.image_paths) == 0:
            self.image_paths = glob.glob(os.path.join(root_dir, '**', '*.png'), recursive=True)
            
        self.to_tensor = T.Compose([
            T.Resize(image_size),
            T.CenterCrop(image_size),
            T.ToTensor()
        ])
        
    def __len__(self):
        return len(self.image_paths)
        
    def __getitem__(self, idx):
        path = self.image_paths[idx]
        with Image.open(path) as img:
            clean = self.to_tensor(img.convert('RGB'))
            
        degraded = clean.clone()
        # Apply degradations randomly
        if random.random() < 0.5:
            degraded = apply_low_light(degraded, severity=random.uniform(0.2, 0.6))
        if random.random() < 0.5:
            degraded = apply_gaussian_blur(degraded, kernel_size=5, sigma=random.uniform(0.5, 2.0))
        if random.random() < 0.7:
            degraded = add_gaussian_noise(degraded, std=random.uniform(0.05, 0.25))
            
        return degraded, clean

In [ ]:
# PSNR & SSIM Evaluation helper functions
def batch_psnr(img1, img2):
    mse = torch.mean((img1 - img2) ** 2, dim=[1, 2, 3])
    psnr = 10.0 * torch.log10(1.0 / (mse + 1e-8))
    return torch.mean(psnr).item()

In [ ]:
def train_restoration_model(model, train_loader, val_loader, optimizer, criterion, epochs=10, save_name='model.pth'):
    model = model.to(DEVICE)
    best_val_psnr = 0.0
    
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for degraded, clean in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]"):
            degraded, clean = degraded.to(DEVICE), clean.to(DEVICE)
            optimizer.zero_grad()
            output = model(degraded)
            loss = criterion(output, clean)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        train_loss /= len(train_loader)
        
        # Validation
        model.eval()
        val_psnr = 0.0
        with torch.no_grad():
            for degraded, clean in val_loader:
                degraded, clean = degraded.to(DEVICE), clean.to(DEVICE)
                restored = model(degraded)
                val_psnr += batch_psnr(restored, clean)
        val_psnr /= len(val_loader)
        
        print(f"Epoch {epoch}: Train Loss = {train_loss:.4f} | Val PSNR = {val_psnr:.2f} dB")
        
        if val_psnr > best_val_psnr:
            best_val_psnr = val_psnr
            save_path = os.path.join(WEIGHTS_DIR, save_name)
            torch.save(model.state_dict(), save_path)
            print(f" --> Saved best checkpoint to {save_path}")
            
    print(f"Training Complete! Best Val PSNR: {best_val_psnr:.2f} dB")